# 2. Compare the prompt stages

Optional experiment after [the normal run notebook](01_run_extraction.ipynb) works. This runs the **same full F16/BF16 MedGemma 27B model**, notes, outcomes, and sampling settings across five cumulative prompt stages. It compares prompts, not different models.

Follow [Ollama setup](../docs/ollama_setup.md) first. Use the project Python kernel on Windows or Linux. Five stages cost approximately five single-stage runs. This notebook does not install software, mount Drive, or require Colab.

| Stage | Change from the previous stage |
| --- | --- |
| `0` | Preserved baseline prompt |
| `1` | JSON schema, field order, rule grouping, flat repetition penalty, content retries |
| `2a` | Rubric presence definition and quoted presence |
| `2b` | Evidence field before findings |
| `3` | Episode scope, negation, ordinal cues, unit wording |

**More accepted findings is not necessarily better.** Stage 3 may correctly reject proposals that earlier stages accepted. Choose a stage using human review, not a ranking of grounding counters.

## 1. Locate the project

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys
from datetime import datetime, timezone

ROOT = next((p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents)
             if (p / 'scripts/experiments/medgemma_extraction.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the cloned st_jude repository.')
sys.path.insert(0, str(ROOT / 'scripts'))
from experiments.review_results import export_reviews

SCRIPT = ROOT / 'scripts/experiments/medgemma_extraction.py'
print(f'Project: {ROOT}\nPython: {sys.executable}')

## 2. Configure the comparison
All stages share these settings. Start with two notes for a hardware check. Keep concurrency at one for repeatability measurements. The fixed sampling seed inside the harness ensures the same cases are selected for every stage. Stage 2b/3 use a larger default completion budget for their evidence field; the budget is recorded in each result.

In [ ]:
MODEL = 'medgemma-27b-f16'
HOST = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
NOTES = 20
# 14 focus outcomes: Pain, Stroke, SS, ACS, Priapism, Chronic Pain, CKD,
# Retinopathy, CD, Depression, TCD Elevation, Asthma, AVN, Leg Ulcer (or 'all').
OUTCOMES = '10,11,12,15,17,21,24,28,29,39,40,47,48,49'
COHORT = 'scd_primary'
STAGES = ['0', '1', '2a', '2b', '3']
REPEAT = 2
CONCURRENCY = 1
NUM_CTX = 16384
TIMEOUT = 600
RUN_DIR = ROOT / 'results' / datetime.now(timezone.utc).strftime('comparison_%Y%m%dT%H%M%S_%fZ')

## 3. Verify the full model
Smaller models, quantized weights, missing digests, and unsuccessful synthetic generations stop execution before patient notes are sent.

In [ ]:
base_command = [sys.executable, str(SCRIPT), '--model', MODEL, '--host', HOST,
                '--timeout', str(TIMEOUT), '--num-ctx', str(NUM_CTX)]
subprocess.run(base_command + ['--check-model'], cwd=ROOT, check=True)

## 4. Run each stage in order
Each successful stage is saved immediately. A failed subprocess stops the comparison; existing outputs are not mistaken for a successful rerun. To rerun everything, create a new `RUN_DIR` in the configuration cell.

In [ ]:
results = {}
for prompt_stage in STAGES:
    out_path = RUN_DIR / f'stage_{prompt_stage}.json'
    command = base_command + [
        '--notes', str(NOTES), '--outcomes', OUTCOMES, '--cohort', COHORT,
        '--stratify', '--holdout-frac', '0.25', '--repeat', str(REPEAT),
        '--concurrency', str(CONCURRENCY), '--prompt-stage', prompt_stage,
        '--out', str(out_path),
    ]
    subprocess.run(command, cwd=ROOT, check=True)
    results[prompt_stage] = json.loads(out_path.read_text(encoding='utf-8'))

digests = {data['provenance']['model_digest'] for data in results.values()}
cohorts = {tuple(r['patient_uid'] for r in data['detailed_records']) for data in results.values()}
if len(digests) != 1 or len(cohorts) != 1:
    raise RuntimeError('The model or selected notes changed during the comparison; do not pool it.')

## 5. Compare automatic counters
`grounded_%` uses quoted proposals as its denominator, not all proposals. These are behavior checks, not clinical precision or recall. Inspect per-outcome distributions in each JSON and use the review sheets below.

In [ ]:
print(f"{'stage':<7} {'accepted':>9} {'grounded_%':>11} {'bad_json':>9} {'seconds':>9}")
for prompt_stage, data in results.items():
    metrics = data['automated_metrics']
    print(f"{prompt_stage:<7} {metrics['accepted']:>9} "
          f"{metrics['quote_verified_pct_of_quoted']:>11.1f} "
          f"{metrics['unparseable_replies']:>9} "
          f"{data['profiling']['total_wall_clock_sec']:>9.1f}")

## 6. Review every stage using its own saved evidence
Each stage gets separate hand-check, conflict, absence, and refuted-presence CSVs, identified by the source result hash. Never copy labels between stages without checking the actual proposal. Review `supports_value` as `y/n`; precision is `y / (y + n)`. Review five full notes for omissions. Blank/empty sheets do not imply correctness.

Existing review sheets are protected from overwrite. For a second independent reviewer, use the export CLI with a different `--output-dir`.

In [ ]:
for prompt_stage in STAGES:
    paths = export_reviews(RUN_DIR / f'stage_{prompt_stage}.json')
    print(f'Stage {prompt_stage}: {paths["handcheck"].parent}')

## 7. Save the comparison
The ZIP includes all stage results and worksheets. Recreate it after editing the CSVs. Share only with authorized collaborators and keep patient-level outputs out of Git.

In [ ]:
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print(f'Artifacts: {RUN_DIR}\nArchive: {archive}')